In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from train import Head, MultiHeadAttention, FeedForward, Block, GPTLanguageModel

# same hyperparameters as train.py
block_size = 256
n_embd = 512
n_head = 8
n_layer = 6
dropout = 0.2
device = 'cpu'

# load checkpoint
checkpoint = torch.load('gpt_checkpoint.pt', map_location='cpu', weights_only=False)
stoi = checkpoint['stoi']
itos = checkpoint['itos']
vocab_size = len(stoi)

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# load model
model = GPTLanguageModel()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# generate
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


Cominius, and be project'st face, not to be general:
Let me trust I know thee report of thy party,
And make the drownfall of Claudio's valour.
Rage, that doth frown and hot, and on earth th't
Most oble true sins, it may not stumblector view;
For thou love'st know thou fonfust: mark me from him.

GLOUCESTER:
Well met unborn, and be your wife should call you of withose
Edward proming with us.

BISHOP OF ELY:
Thou wilt had lived morrow, so were so at lawful,
With death's hopes from a beggast of blo


KV Cache

In [3]:
from kv_cache import CacheHead, CachedBlock, CachedGPT

import time

context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Naive
start = time.time()
model.generate(context, max_new_tokens=500)
naive_time = time.time() - start
print(f"Naive: {naive_time:.2f}s | {500/naive_time:.1f} tok/s")

# Cached
cached_model = CachedGPT()
cached_model.load_state_dict(checkpoint['model_state_dict'])
cached_model.eval()

start = time.time()
cached_model.generate_with_cache(context, max_new_tokens=500)
cached_time = time.time() - start
print(f"Cached: {cached_time:.2f}s | {500/cached_time:.1f} tok/s")

print(f"Speedup: {naive_time/cached_time:.2f}x")

Naive: 51.86s | 9.6 tok/s
Cached: 14.61s | 34.2 tok/s
Speedup: 3.55x
